# Notebook 06 — Stage 3: low-stress share search, outer loop, and the single held-out evaluation

*Thesis §Share Update Procedure, §Convergence.*

With $A_{low}$, $A_{high}$ frozen (Notebook 05), iterate:

1. $\pi^{(j)}_{low}=\arg\min$ mean GEH on the **training** stations over a 101-point grid;
2. re-estimate $t^{(j)}$ by **training-only** constrained GLS (Bell 1991) at $A(\pi^{(j)})$.

Stop when $\pi$ stops moving ($|\Delta\pi|<0.01$). Because the grid spacing equals the tolerance, two
adjacent values can alternate; that case is detected and the member with the lower training mean GEH is
taken (deterministic).

**Leakage firewall.**
1. Everything up to Section F uses the 53 training stations only (`train_mask.npy`).
2. Section F freezes $\pi^*$ and $t^*$ to `stage3_final.json` **before** the held-out mask is loaded.
3. Section G loads the held-out mask through `tc.load_test_mask` (locked until step 2 happens, and logged)
   and scores the 24 held-out stations **once**.
4. Everything after Section G is descriptive: it reports the frozen $\pi^*$, $t^*$ and leaves them unchanged.

**Inputs:** Notebook 04 and Notebook 05 outputs. **Outputs:** `od_final_t.csv`, `stage3_trace.csv`,
`pi_objective_curves.csv`, `stage3_final.json`, `stage3_link_fit.csv`, `stage3_metrics.json`,
`results_top_flow_links.csv`, `results_top_error_links.csv`, and the Stage-3 text report.


## Setup and configuration

Imports, shared thesis style, and the full parameter table (with literature sources) printed for the record.


In [2]:
import json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
import thesis_common as tc

tc.set_style()
NB = "06"
print("Run stamp:", tc.RUN_STAMP)
display(tc.config_table())     # all model parameters + their literature sources


Run stamp: 20261104_2255


,parameter,value,source
0,v_flat (km/h),21.6,Parkin & Rotheram (2010)
1,kappa_up (km/h per %),1.44,Parkin & Rotheram (2010)
2,kappa_down (km/h per %),0.86,Parkin & Rotheram (2010)
3,v_min (km/h),6.0,Parkin & Rotheram (2010)
4,E_low threshold (LTS <=),1,Mekuria et al. (2012); conservative choice
5,lambda_stress,2.4,"Broach et al. (2012) commute, 20-30k AADT (+140%)"
6,k paths,3,modelling choice; swept 1-5 (NB09b)
7,pi_low0,0.81,Dill & McNeil (2013): 56/69
8,pi grid / tol / max outer,101 / 0.01 / 25,method choice
9,sigma0^2,10.0,no literature value; train-only CV check in NB09a


# A — Inputs (training stations only)


Load the frozen class matrices, the calibrated demand, the monitored-link metadata and the count survey;
build the count-dispersion matrix $W$; and report the network dimensions and class identifiability.


In [3]:
# Frozen class incidence matrices (NB05) + the neutral matrix (NB04).
A_low  = sparse.load_npz(tc.DATA_PROCESSED / "A_low.npz").tocsr()
A_high = sparse.load_npz(tc.DATA_PROCESSED / "A_high.npz").tocsr()
A_neu  = sparse.load_npz(tc.DATA_PROCESSED / "A_neutral.npz").tocsr()

# Calibrated demand, monitored links, and the count-station snap table.
cal    = pd.read_csv(tc.DATA_PROCESSED / "od_calibrated_t.csv").sort_values("pair_idx").reset_index(drop=True)
links  = pd.read_csv(tc.DATA_PROCESSED / "A_neutral_links.csv").sort_values("link_idx").reset_index(drop=True)
snap   = pd.read_csv(tc.DATA_PROCESSED / "count_link_snap.csv")

L, P   = len(links), len(cal)
v_obs  = links["v_obs"].to_numpy(float)
n_seed = cal["n_seed"].to_numpy(float); t_cal = cal["t"].to_numpy(float)
train  = tc.load_train_mask(L)

# Shapes must agree across the three matrices, or an upstream notebook is out of sync.
assert A_low.shape == A_high.shape == A_neu.shape == (L, P), "matrix shapes disagree - re-run NB04/NB06"

# Count-dispersion matrix W (per-link variance of round-mean peak counts over the window).
survey = tc.load_survey()
W, n_own = tc.build_W(survey, tc.site_to_link(snap, links), L)

print(f"{L} monitored links, {P:,} O-D pairs | training stations {int(train.sum())}")
print(f"W: {n_own}/{L} links with their own variance; median sd {np.sqrt(np.median(W)):.1f} trips/h")
print(f"identifiability |A_low-A_high|/|A_high| = {tc.a_diff(A_low, A_high):.0%}")


77 monitored links, 1,850 O-D pairs | training stations 53
W: 77/77 links with their own variance; median sd 10.8 trips/h
identifiability |A_low-A_high|/|A_high| = 120%


# B — Earlier stages on the training stations (the results ladder)


The fit at each earlier rung — raw census seed, Stage-1 GLS demand, and Stage-2 two-class assignment at
$\pi^{(0)}$ — on the training stations, so the Stage-3 gain can be read against them.


In [4]:
# Training-station fit at each rung below Stage 3, for the results ladder.
ladder_train = {
    "Seed $n$, neutral AON":            tc.metrics(np.asarray(A_neu.dot(n_seed)).ravel()[train], v_obs[train]),
    "Stage 1: GLS $t$, neutral AON":    tc.metrics(np.asarray(A_neu.dot(t_cal)).ravel()[train], v_obs[train]),
    "Stage 2: two-class at $\\pi^{(0)}$": tc.metrics(tc.flow(A_low, A_high, tc.PI_LOW0, t_cal)[train], v_obs[train]),
}
for k_, m in ladder_train.items():
    print(f"  {k_:<36} {tc.fmt_m(m)}")


  Seed $n$, neutral AON                meanGEH 9.00 | GEH<5 24.5% | RMSE 128.0 | R2 0.187 (CoD -0.190) | bias -29% | n=53
  Stage 1: GLS $t$, neutral AON        meanGEH 6.35 | GEH<5 54.7% | RMSE 107.4 | R2 0.391 (CoD 0.162) | bias -49% | n=53
  Stage 2: two-class at $\pi^{(0)}$    meanGEH 6.57 | GEH<5 45.3% | RMSE 89.5 | R2 0.557 (CoD 0.418) | bias -33% | n=53


# C — Outer loop (training stations only)


Iterate the share search and the GLS re-estimation until $\pi$ stops moving, then report the objective
optima and the near-optimal band at convergence.


In [5]:
# Objective optima at the Stage-1 demand (the first search, before any re-estimation).
cur0 = tc.objective_curves(A_low, A_high, t_cal, v_obs, train)
print(f"first search at Stage-1 t: mean-GEH optimum {cur0['pi_meangeh']:.2f} | "
      f"RMSE optimum {cur0['pi_rmse']:.2f} | GEH-pass optimum {cur0['pi_gehpass']:.2f}")

# Run the outer loop (share search + training-only GLS) to convergence.
res = tc.outer_loop(A_low, A_high, t_cal, n_seed, v_obs, W, train, verbose=True)
pi_star, t_star, trace = res["pi_star"], res["t_star"], res["trace"]
print(f"\nCONVERGED after {res['n_iter']} outer iterations: {res['reason']}")
print(f"pi_low* = {pi_star:.2f}   (stress-tolerant share {1-pi_star:.2f})")

# Objective optima at the converged demand t*, plus the share band within 2% of the minimum mean GEH.
curS = tc.objective_curves(A_low, A_high, t_star, v_obs, train)
band = curS["grid"][curS["mean_geh"] <= curS["mean_geh"].min() * 1.02]
print(f"at t*: mean-GEH optimum {curS['pi_meangeh']:.2f} | RMSE optimum {curS['pi_rmse']:.2f} | "
      f"GEH-pass optimum {curS['pi_gehpass']:.2f} | pi within 2% of min mean GEH: {band.min():.2f}-{band.max():.2f}")


first search at Stage-1 t: mean-GEH optimum 0.33 | RMSE optimum 0.72 | GEH-pass optimum 0.57
  outer  1: pi=0.33  |dpi|=0.480  train meanGEH 5.560  GEH<5 58.5%  sum t 3,760
  outer  2: pi=0.45  |dpi|=0.120  train meanGEH 5.269  GEH<5 62.3%  sum t 3,784
  outer  3: pi=0.52  |dpi|=0.070  train meanGEH 5.125  GEH<5 62.3%  sum t 3,804
  outer  4: pi=0.57  |dpi|=0.050  train meanGEH 5.048  GEH<5 62.3%  sum t 3,820
  outer  5: pi=0.61  |dpi|=0.040  train meanGEH 5.006  GEH<5 64.2%  sum t 3,833
  outer  6: pi=0.63  |dpi|=0.020  train meanGEH 4.987  GEH<5 64.2%  sum t 3,840
  outer  7: pi=0.64  |dpi|=0.010  train meanGEH 4.977  GEH<5 64.2%  sum t 3,844
  outer  8: pi=0.64  |dpi|=0.000  train meanGEH 4.977  GEH<5 64.2%  sum t 3,844

CONVERGED after 8 outer iterations: |dpi| < 0.01
pi_low* = 0.64   (stress-tolerant share 0.36)
at t*: mean-GEH optimum 0.64 | RMSE optimum 0.77 | GEH-pass optimum 0.76 | pi within 2% of min mean GEH: 0.57-0.72


# D — Training-only diagnostics that justify the method


**Diagnostic A:** how many links carry a distinct routing signal between the two classes (so $\pi$ is
estimable). **Diagnostic B:** how stable the optimal share is across prior-dispersion values, using
training data only. (The full $\sigma_0^2$ robustness sweep is Notebook 09a.)


In [6]:
# Diagnostic A: how many links (and how many training links) carry a non-zero low/high routing difference.
row = np.asarray(abs(A_low - A_high).sum(axis=1)).ravel()
print(f"DIAG A  links carrying pi signal: {int((row>1e-9).sum())}/{L} "
      f"(training {int(((row>1e-9)&train).sum())}/{int(train.sum())})")

# Diagnostic B: optimal share by objective across a sigma0^2 sweep (training only); spans show stability.
diagB = []
for s2 in [1.0, 2.0, 3.0, 5.0, 7.5, 10.0]:
    ts = tc.solve_gls(tc.composite(A_low, A_high, pi_star), n_seed, v_obs, W, train, s2)
    c = tc.objective_curves(A_low, A_high, ts, v_obs, train)
    diagB.append(dict(sigma2=s2, pi_meangeh=c["pi_meangeh"], pi_rmse=c["pi_rmse"], pi_gehpass=c["pi_gehpass"]))
diagB = pd.DataFrame(diagB)
print("DIAG B  optimum by objective across sigma0^2 (train only):")
print(diagB.to_string(index=False))
print(f"        spans: mean GEH {np.ptp(diagB.pi_meangeh):.2f} | RMSE {np.ptp(diagB.pi_rmse):.2f} | "
      f"GEH-pass {np.ptp(diagB.pi_gehpass):.2f}")


DIAG A  links carrying pi signal: 72/77 (training 49/53)
DIAG B  optimum by objective across sigma0^2 (train only):
 sigma2  pi_meangeh  pi_rmse  pi_gehpass
    1.0        0.66     0.71        0.72
    2.0        0.64     0.76        0.76
    3.0        0.63     0.77        0.73
    5.0        0.63     0.78        0.86
    7.5        0.64     0.77        0.77
   10.0        0.64     0.77        0.76
        spans: mean GEH 0.03 | RMSE 0.07 | GEH-pass 0.14


# E — Diagnostic figures (training stations only)


Three diagnostics: outer-loop convergence over iterations, the share objective at the first vs final
iteration, and objective stability across prior dispersion.


In [7]:
# Figure 1: outer-loop convergence (pi_low, training mean GEH, and total demand over iterations).
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
axes[0].plot(trace.outer, trace.pi_low, "o-", color="black")
axes[0].axhline(pi_star, color=tc.COL["low"], ls="--", lw=1, label=f"$\\pi^*$={pi_star:.2f}")
axes[0].axhline(tc.PI_LOW0, color=tc.COL["neutral"], ls=":", lw=1, label=f"$\\pi^{{(0)}}$={tc.PI_LOW0}")
axes[0].set_xlabel("outer iteration"); axes[0].set_ylabel(r"$\pi_{low}$"); axes[0].legend()

axes[1].plot(trace.outer, trace.train_mean_geh, "o-", color=tc.COL["train"])
axes[1].set_xlabel("outer iteration"); axes[1].set_ylabel("train mean GEH")

axes[2].plot(trace.outer, trace.t_sum, "o-", color=tc.COL["neutral"])
axes[2].axhline(n_seed.sum(), color=tc.COL["neutral"], ls=":", lw=1, label="seed total")
axes[2].set_xlabel("outer iteration"); axes[2].set_ylabel(r"$\sum t$ (trips/h)"); axes[2].legend()

fig.suptitle("Outer-loop convergence (training stations only)", y=1.03)
tc.save_diag(fig, NB, "convergence_train")

# Figure 2: the share objective at the first iteration vs the final converged demand.
fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.plot(cur0["grid"], cur0["mean_geh"], color=tc.COL["neutral"], lw=1.2, ls="--", label="iteration 1 (Stage-1 $t$)")
ax.plot(curS["grid"], curS["mean_geh"], color="black", lw=2, label="final ($t^*$)")
ax.axvline(pi_star, color=tc.COL["low"], ls="--", lw=1)
ax.set_xlabel(r"$\pi_{low}$"); ax.set_ylabel("train mean GEH"); ax.legend()
ax.set_title("Share objective, first vs final outer iteration")
tc.save_diag(fig, NB, "objective_first_vs_final")

# Figure 3: optimal pi_low by objective across prior dispersion (training only).
fig, ax = plt.subplots(figsize=(7.5, 4.3))
for col, lab, c in [("pi_meangeh", "mean GEH", "black"), ("pi_rmse", "RMSE", tc.COL["neutral"]),
                    ("pi_gehpass", "GEH-pass", tc.COL["test"])]:
    ax.plot(diagB.sigma2, diagB[col], "o-", color=c, label=lab)
ax.set_xlabel(r"$\sigma_0^2$"); ax.set_ylabel(r"optimal $\pi_{low}$"); ax.set_ylim(0, 1); ax.legend()
ax.set_title("Objective stability across prior dispersion (train only)")
tc.save_diag(fig, NB, "objective_stability_train")


  saved diagnostic     06_convergence_train.png
  saved diagnostic     06_objective_first_vs_final.png
  saved diagnostic     06_objective_stability_train.png


# F — Freeze $\pi^*$ and $t^*$ (before any held-out data is loaded)


Serialise and lock the converged parameters to `stage3_final.json` (with a hash of $t^*$) and export the
final demand table, the outer-loop trace and the objective curves — all **before** the held-out mask is
touched in Section G.


In [8]:
# Freeze pi* and t* (with run metadata) the moment they are fixed, before any held-out data is loaded.
final = tc.write_final(pi_star, t_star, extra=dict(n_iter=res["n_iter"], reason=res["reason"],
                                                  sigma2=tc.SIGMA2, lam=tc.STRESS_PENALTY, k=tc.K_PATHS))

# Export the final demand table, the outer-loop trace, and the objective curves (first vs final).
cal.assign(t_final=t_star).to_csv(tc.DATA_PROCESSED / "od_final_t.csv", index=False)
trace.to_csv(tc.DATA_PROCESSED / "stage3_trace.csv", index=False)
pd.DataFrame({"pi": curS["grid"], "mean_geh_final": curS["mean_geh"], "rmse_final": curS["rmse"],
              "geh_pass_final": curS["geh_pass"], "mean_geh_iter1": cur0["mean_geh"]}).to_csv(
    tc.DATA_PROCESSED / "pi_objective_curves.csv", index=False)

print("frozen:", json.dumps(final, indent=1))


frozen: {
 "pi_star": 0.64,
 "t_star_sha1": "4ca6efa646bda9854ac5d1421183d3c5645ed227",
 "t_star_sum": 3843.9284962489132,
 "frozen_at": "2026-11-04T22:55:43",
 "n_iter": 8,
 "reason": "|dpi| < 0.01",
 "sigma2": 10.0,
 "lam": 2.4,
 "k": 3
}


# G — The single held-out evaluation


Load the held-out mask (now unlocked, and logged), score the 24 held-out stations once at the frozen
$(\pi^*, t^*)$, build the link-level fit table with station metadata, and write `stage3_metrics.json`.

*Provenance:* this cell produces **thesis Table 4.1** (training vs held-out mean GEH / GEH-pass / RMSE /
$R^2$), saved to `stage3_metrics.json`; the ladder and trace go to the Stage-3 report.


In [9]:
# Load the held-out mask (locked until pi* was frozen above; the access is logged).
test = tc.load_test_mask(L, "final evaluation at frozen (pi*, t*)", "NB07")

# Final composite flow at (pi*, t*), then the single training vs held-out evaluation.
f_final = tc.flow(A_low, A_high, pi_star, t_star)
m_tr = tc.metrics(f_final[train], v_obs[train])
m_te = tc.metrics(f_final[test],  v_obs[test])
print(f"TRAIN    (n={m_tr['n']}): {tc.fmt_m(m_tr)}")
print(f"HELD-OUT (n={m_te['n']}): {tc.fmt_m(m_te)}")
print(f"guideline: GEH<5 on >= {tc.GEH_GUIDE:.0f}% of sites -> held-out {m_te['geh_pass']:.1f}%")

# Link-level fit table: modelled flows per stage, GEH, residual, split and station labels.
names = tc.link_names(links, snap)
fit = links.assign(f_final=f_final, f_pi0=tc.flow(A_low, A_high, tc.PI_LOW0, t_cal),
                   f_stage1=np.asarray(A_neu.dot(t_cal)).ravel(),
                   geh_final=tc.geh(f_final, v_obs), resid=f_final - v_obs,
                   split=np.where(train, "train", "test"),
                   location=[names.get(i, (f"link {i}", None))[0] for i in range(L)],
                   SiteID=[names.get(i, (None, None))[1] for i in range(L)])
fit.to_csv(tc.DATA_PROCESSED / "stage3_link_fit.csv", index=False)

# Monitored-flow split between the two classes at pi*.
fl = float((pi_star * A_low).dot(t_star).sum()); fh = float(((1-pi_star) * A_high).dot(t_star).sum())

# Bundle every headline number into stage3_metrics.json for the report-figures notebook.
metrics_out = dict(pi_star=pi_star, n_iter=res["n_iter"], reason=res["reason"],
                   train=m_tr, test=m_te, ladder_train=ladder_train,
                   flow_split_low_pct=100*fl/(fl+fh), a_diff=tc.a_diff(A_low, A_high),
                   sum_n=float(n_seed.sum()), sum_t_stage1=float(t_cal.sum()), sum_t_final=float(t_star.sum()),
                   pi_rmse=curS["pi_rmse"], pi_gehpass=curS["pi_gehpass"])
(tc.DATA_PROCESSED / "stage3_metrics.json").write_text(json.dumps(metrics_out, indent=2))


TRAIN    (n=53): meanGEH 4.98 | GEH<5 64.2% | RMSE 82.4 | R2 0.617 (CoD 0.506) | bias -34% | n=53
HELD-OUT (n=24): meanGEH 5.62 | GEH<5 41.7% | RMSE 57.4 | R2 0.545 (CoD 0.196) | bias -9% | n=24
guideline: GEH<5 on >= 85% of sites -> held-out 41.7%


1774

# H — Final-fit tables

Descriptive only — every table below reports the frozen model ($\pi^*$, $t^*$).


The ten busiest stations by observed volume and the ten with the largest GEH error, exported for the
results tables and printed here.


In [10]:
# Ten busiest stations by observed volume, with modelled flow per stage.
top_flow = fit.sort_values("v_obs", ascending=False).head(10)[
    ["SiteID", "location", "split", "v_obs", "f_stage1", "f_pi0", "f_final", "geh_final"]].round(1)

# Ten largest-GEH stations (the worst residuals at the frozen model).
top_err = fit.sort_values("geh_final", ascending=False).head(10)[
    ["SiteID", "location", "split", "v_obs", "f_final", "resid", "geh_final"]].round(1)

top_flow.to_csv(tc.FIG_REPORT / "results_top_flow_links.csv", index=False)
top_err.to_csv(tc.FIG_REPORT / "results_top_error_links.csv", index=False)

print("TOP-FLOW stations (tab:topflow)"); print(top_flow.to_string(index=False))
print("\nLARGEST-GEH stations (tab:toperr)"); print(top_err.to_string(index=False))


TOP-FLOW stations (tab:topflow)
 SiteID                      location split  v_obs  f_stage1  f_pi0  f_final  geh_final
     78 Murray St / Union St Cycleway train  510.3     468.9  466.8    418.1        4.3
     72       Flinders St / Oxford St train  506.2     125.6  397.9    397.8        5.1
     74       Moore Park Rd / Lang Rd train  376.7       3.2    3.9      4.4       27.0
     12  Liverpool St / Wentworth Ave train  335.0      77.2  205.9    222.7        6.7
     66      Cleveland St / Anzac Pde train  314.7      75.6   74.0     80.0       16.7
     11             Kent St / King St train  311.5     170.6  415.3    310.7        0.0
     65   Moore Park Rd / Flinders St train  282.0      24.7   25.5     26.0       20.6
     21           Harbour Br Cycleway train  222.7     291.1  291.1    291.7        4.3
     14                  Liverpool St  test  211.8     111.3  231.7    240.0        1.9
     80           Bourke St / Cres St  test  208.5     110.1  165.0    153.4        4.1


# I — External plausibility (thesis §External Plausibility Benchmarking)


Scale the modelled peak-hour demand to a daily basis using the survey's all-day hour columns (or a
manual peak-hour share) and compare against external daily benchmarks. In the final run the survey has
only the three peak hours, so this check is skipped unless `PEAK_HOUR_SHARE` is set from a cited source.


In [11]:
# Optional external plausibility check: needs a peak-hour share (manual, or derived from all-day survey columns).
PEAK_HOUR_SHARE = None           # e.g. 0.11 (fraction of daily trips in the mean AM-peak hour)

hour_cols = [c for c in survey.columns if c.startswith("Time_")]

# Derive the peak-hour share empirically only if the survey carries enough all-day hour columns.
if PEAK_HOUR_SHARE is None and len(hour_cols) >= 20:
    s = survey[survey["Year"] == 2025]
    allday = s[hour_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1)
    PEAK_HOUR_SHARE = float((s["peak"] / allday.replace(0, np.nan)).median())
    print(f"peak-hour share from the survey's {len(hour_cols)} hour columns: {PEAK_HOUR_SHARE:.3f}")

# Scale to a daily equivalent and compare to the external benchmarks if a share is available.
if PEAK_HOUR_SHARE:
    daily = t_star.sum() / PEAK_HOUR_SHARE
    internal = t_star[cal["movement"].eq("inbound_internal").to_numpy()].sum() / PEAK_HOUR_SHARE
    print(f"modelled peak-hour demand {t_star.sum():,.0f} -> ~{daily:,.0f} trips/day "
          f"(internal-only ~{internal:,.0f})")
    print("benchmarks (thesis): ~48,000 (cycle-km), ~23,000 internal + ~29,000 external (HTS), "
          "~28,000 (ML), ~3,500 (2016 JTW commuters)")
else:
    print(f"survey has {len(hour_cols)} hour columns (needs all-day data) -> set PEAK_HOUR_SHARE "
          "from an external reference source to run this check.")


survey has 6 hour columns (needs all-day data) -> set PEAK_HOUR_SHARE from an external reference source to run this check.


# J — Count-station reconciliation and report


Confirm the 2025 survey coverage, print any unmatched count stations from the snap audit, and assemble
the full Stage-3 text report (parameters, convergence trace, results ladder, and the single held-out
evaluation) — this notebook's end-of-run key-results summary.


In [12]:
# Coverage check: 2025 stations surveyed in both rounds, against the monitored train/held-out split.
s25 = survey[survey["Year"] == 2025]
both = set(s25[(s25.Month == "March") & s25.peak.notna()].SiteID) & set(s25[(s25.Month == "October") & s25.peak.notna()].SiteID)
print(f"2025 stations with both rounds: {len(both)} | monitored links used: {L} "
      f"({int(train.sum())} training / {int(test.sum())} held-out)")

# If the snap audit exists, list any stations that failed to snap and why.
audit = tc.DATA_PROCESSED / "count_snap_audit.csv"
if audit.exists():
    a = pd.read_csv(audit); print(a[~a["matched"].astype(bool)][["SiteID", "reason"]].to_string(index=False))

# Assemble and write the Stage-3 report (parameters, convergence, ladder, held-out, and the outer trace).
report = f"""07 - Stage 3 report ({tc.RUN_STAMP})
{'='*60}
Objective: train mean GEH over a {tc.PI_GRID}-point grid; GLS on training rows only.
pi_low0 = {tc.PI_LOW0}; sigma0^2 = {tc.SIGMA2}; lambda = {tc.STRESS_PENALTY}; k = {tc.K_PATHS}
Converged: {res['reason']} after {res['n_iter']} iterations
pi_low* = {pi_star:.2f}  (RMSE optimum {curS['pi_rmse']:.2f}, GEH-pass optimum {curS['pi_gehpass']:.2f};
          within 2% of min mean GEH: {band.min():.2f}-{band.max():.2f})
Identifiability |A_low-A_high|/|A_high| = {tc.a_diff(A_low, A_high):.0%}
Monitored-flow split at pi*: low {100*fl/(fl+fh):.0f}% / high {100*fh/(fl+fh):.0f}% (demand share {100*pi_star:.0f}%)
Demand: seed {n_seed.sum():,.0f} -> Stage 1 {t_cal.sum():,.0f} (x{t_cal.sum()/n_seed.sum():.2f}) -> final {t_star.sum():,.0f} (x{t_star.sum()/n_seed.sum():.2f})
  median t/n: Stage 1 {np.median(t_cal[n_seed>0]/n_seed[n_seed>0]):.2f}, final {np.median(t_star[n_seed>0]/n_seed[n_seed>0]):.2f}

Ladder (training stations):
""" + "\n".join(f"  {k_:<36} {tc.fmt_m(m)}" for k_, m in ladder_train.items()) + f"""
  Stage 3 at pi*                       {tc.fmt_m(m_tr)}
Held-out (single evaluation):
  Stage 3 at pi*                       {tc.fmt_m(m_te)}

Outer-loop trace:
{trace.round(3).to_string(index=False)}
"""
(tc.REPORTS / "07_stage3_report.txt").write_text(report, encoding="utf-8")
print(report)


2025 stations with both rounds: 78 | monitored links used: 77 (53 training / 24 held-out)
 SiteID                  reason
    110 never_snapped_no_coords
07 - Stage 3 report (20261104_2255)
Objective: train mean GEH over a 101-point grid; GLS on training rows only.
pi_low0 = 0.81; sigma0^2 = 10.0; lambda = 2.4; k = 3
Converged: |dpi| < 0.01 after 8 iterations
pi_low* = 0.64  (RMSE optimum 0.77, GEH-pass optimum 0.76;
          within 2% of min mean GEH: 0.57-0.72)
Identifiability |A_low-A_high|/|A_high| = 120%
Monitored-flow split at pi*: low 71% / high 29% (demand share 64%)
Demand: seed 4,264 -> Stage 1 3,714 (x0.87) -> final 3,844 (x0.90)
  median t/n: Stage 1 1.00, final 1.00

Ladder (training stations):
  Seed $n$, neutral AON                meanGEH 9.00 | GEH<5 24.5% | RMSE 128.0 | R2 0.187 (CoD -0.190) | bias -29% | n=53
  Stage 1: GLS $t$, neutral AON        meanGEH 6.35 | GEH<5 54.7% | RMSE 107.4 | R2 0.391 (CoD 0.162) | bias -49% | n=53
  Stage 2: two-class at $\pi^{(0)}$    